In [2]:
import pandas as pd
import numpy as np

from scipy.sparse import csr_matrix

from sklearn.metrics.pairwise import cosine_similarity

import pickle

print("Libraries Loaded")

Libraries Loaded


In [3]:
recommendation_df = pd.read_csv(
    r"E:\Reel-Recommendation-Engine\data\processed\recommendation_dataset.csv"
)

print(recommendation_df.shape)

(1414622, 118)


In [4]:
cf_df = recommendation_df[
    [
        "user_id",
        "video_id",
        "recommendation_score"
    ]
].copy()

cf_df.head()

,user_id,video_id,recommendation_score
0,0,1527,0.058969
1,0,7405,0.007104
2,0,6026,0.052402
3,1,6354,0.009077
4,1,3645,0.018201


In [5]:
print("Unique Users :", cf_df["user_id"].nunique())

print("Unique Videos :", cf_df["video_id"].nunique())

print("Interactions :", len(cf_df))

Unique Users : 27077
Unique Videos : 7551
Interactions : 1414622


In [6]:
interaction_matrix = cf_df.pivot_table(
    index="user_id",
    columns="video_id",
    values="recommendation_score",
    fill_value=0
)

In [7]:
print(interaction_matrix.shape)

interaction_matrix.head()

(27077, 7549)


video_id,0,1,2,3,4,5,6,7,8,9,...,7573,7574,7575,7576,7577,7578,7579,7580,7581,7582
user_id,,,,,,,,,,,,,,,,,,,,,
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
sparse_matrix = csr_matrix(
    interaction_matrix.values
)

print(sparse_matrix.shape)

(27077, 7549)


In [9]:
item_similarity = cosine_similarity(
    sparse_matrix.T
)

In [10]:
print(item_similarity.shape)

(7549, 7549)


In [11]:
video_ids = interaction_matrix.columns

video_to_idx = {
    video: idx
    for idx, video in enumerate(video_ids)
}

idx_to_video = {
    idx: video
    for idx, video in enumerate(video_ids)
}

In [12]:
def recommend_similar_videos(
        video_id,
        top_n=10):

    if video_id not in video_to_idx:
        return "Video Not Found"

    idx = video_to_idx[video_id]

    similarity_scores = list(
        enumerate(item_similarity[idx])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[1:top_n+1]

    recommendations = [
        idx_to_video[i[0]]
        for i in similarity_scores
    ]

    return recommendations

In [13]:
sample_video = interaction_matrix.columns[0]

print("Sample Video:", sample_video)

recommend_similar_videos(sample_video)

Sample Video: 0


[3409, 7580, 4183, 5508, 2374, 4944, 5153, 6503, 1172, 2194]

In [14]:
import os

os.makedirs(
    r"E:\Reel-Recommendation-Engine\models",
    exist_ok=True
)

In [15]:
with open(
    r"E:\Reel-Recommendation-Engine\models\item_similarity.pkl",
    "wb"
) as f:

    pickle.dump(item_similarity, f)

In [16]:
with open(
    r"E:\Reel-Recommendation-Engine\models\interaction_matrix.pkl",
    "wb"
) as f:

    pickle.dump(interaction_matrix, f)

In [17]:
with open(
    r"E:\Reel-Recommendation-Engine\models\video_to_idx.pkl",
    "wb"
) as f:

    pickle.dump(video_to_idx, f)

with open(
    r"E:\Reel-Recommendation-Engine\models\idx_to_video.pkl",
    "wb"
) as f:

    pickle.dump(idx_to_video, f)

print("Collaborative Filtering Models Saved")

Collaborative Filtering Models Saved
